## Data setup (Colab)

This lab uses the Kaggle **digit-recognizer** files `train.csv` and `test.csv`.

**On Colab**, run this in a cell first (needs your Kaggle API token):
```python
!pip install -q kagglehub
import kagglehub
path = kagglehub.competition_download('digit-recognizer')
print(path)
```
The loader cells below find the files automatically at `/kaggle/input/digit-recognizer/`,
`/content/`, or in the current folder (upload them via the Files panel if needed).

In [ ]:
# %% STEP 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)

In [ ]:
# %% STEP 2: load train CSV (Colab + local)
import os
import glob
import pandas as pd

CANDIDATES = [
    'train.csv', 'lab08/train.csv', '/content/train.csv',
    '/kaggle/input/digit-recognizer/train.csv',
] + glob.glob('/kaggle/input/*/train.csv') + glob.glob('/content/*/train.csv')

TRAIN_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)

if TRAIN_PATH is None:
    try:
        import kagglehub
        TRAIN_PATH = os.path.join(kagglehub.competition_download('digit-recognizer'), 'train.csv')
    except Exception as e:
        raise FileNotFoundError(
            'train.csv not found. Upload it, or run:\n'
            "    !pip install -q kagglehub; import kagglehub; kagglehub.competition_download('digit-recognizer')"
        ) from e

print('Using:', TRAIN_PATH)
df = pd.read_csv(TRAIN_PATH)
print('Data shape:', df.shape)
df.head()


In [ ]:
# %% STEP 3: Data Preprocessing
# Label save as y and pixels save as X
y = df.iloc[:, 0].values            
X = df.iloc[:, 1:].values           

# Normalization: pixel values 0-255 to 0-1 range
X = X.astype("float32") / 255.0

# Reshape: 784 -> 28 x 28 x 1 (height, width, channel)
X = X.reshape(-1, 28, 28, 1)

print("X shape:", X.shape, " y shape:", y.shape)

In [ ]:
# %% STEP 4: Training and Validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Validation:", X_val.shape)

In [ ]:
# %% STEP 5: Show some images (data check)
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i].reshape(28, 28), cmap="gray")
    plt.title("Label: " + str(y_train[i]))
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# %% STEP 6: CNN model
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # --- Block 1 ---
    layers.Conv2D(32, (3, 3)),          # Convolution (32 filters, 3x3)
    layers.Activation("relu"),          # ReLU
    layers.MaxPooling2D((2, 2)),        # Max Pooling 2x2

    # --- Block 2 ---
    layers.Conv2D(64, (3, 3)),          # Convolution (64 filters)
    layers.Activation("relu"),          # ReLU
    layers.MaxPooling2D((2, 2)),        # Max Pooling

    # --- Classifier ---
    layers.Flatten(),                   # 2D feature maps -> 1D vector
    layers.Dense(128),                  # Fully Connected layer
    layers.Activation("relu"),
    layers.Dropout(0.3),                # overfitting kam karne ke liye
    layers.Dense(10),                   # 10 classes (digits 0-9)
    layers.Activation("softmax"),       # Softmax -> probabilities
])

model.summary()

In [ ]:
# %% STEP 7: Model compile 
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",   # labels integer (0-9)
    metrics=["accuracy"],
)

In [ ]:
# %% STEP 8: Training
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_val, y_val),
)

In [ ]:
# 9a) Accuracy and Loss graphs on Traning and Validation
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("Loss")
plt.show()

In [ ]:
# %% STEP 10: predict on one test image (Colab + local)
import os
import glob
import numpy as np
import pandas as pd

TEST_CANDIDATES = [
    'test.csv', 'lab08/test.csv', '/content/test.csv',
    '/kaggle/input/digit-recognizer/test.csv',
] + glob.glob('/kaggle/input/*/test.csv') + glob.glob('/content/*/test.csv')

TEST_PATH = next((p for p in TEST_CANDIDATES if os.path.exists(p)), None)
if TEST_PATH is None:
    raise FileNotFoundError('test.csv not found. Upload it or download the digit-recognizer dataset.')
print('Using:', TEST_PATH)

test_df = pd.read_csv(TEST_PATH)
X_test = (test_df.values.astype('float32') / 255.0).reshape(-1, 28, 28, 1)

idx = 5                                    # any image index

probs = model.predict(X_test[idx:idx+1], verbose=0)   # a single image
label = np.argmax(probs)

plt.imshow(X_test[idx].reshape(28, 28), cmap='gray')
plt.title(f'Predicted label: {label}')
plt.axis('off')
plt.show()

print('Predicted label:', label)


In [ ]:
# %% STEP 11: show images.png (Colab + local)
import os
from PIL import Image

IMG_CANDIDATES = ['images.png', 'lab08/images.png', '/content/images.png']
img_path = next((p for p in IMG_CANDIDATES if os.path.exists(p)), None)
if img_path is None:
    import urllib.request
    url = 'https://raw.githubusercontent.com/manaal6/Introduction_To_AI_Lab_Tasks/main/lab08/images.png'
    print('Downloading:', url)
    img_path = 'images.png'
    urllib.request.urlretrieve(url, img_path)
print('Using:', img_path)

img = Image.open(img_path)
plt.imshow(img)
plt.axis('off')
plt.show()
